In [1]:
%pip install -q kaggle
!apt-get update -qq
!apt-get install -y -qq unrar
%pip install -q --upgrade kagglehub

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded.")

Kaggle credentials loaded.


In [3]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get(
    "KAGGLE_USERNAME"
)

os.environ["KAGGLE_KEY"] = userdata.get(
    "KAGGLE_KEY"
)

assert os.environ["KAGGLE_USERNAME"], (
    "KAGGLE_USERNAME is missing from Colab Secrets."
)

assert os.environ["KAGGLE_KEY"], (
    "KAGGLE_KEY is missing from Colab Secrets."
)

print("Kaggle credentials loaded securely.")

Kaggle credentials loaded securely.


In [4]:
import os
from pathlib import Path

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get(
    "KAGGLE_API_TOKEN"
)

import kagglehub

DOWNLOAD_DIR = Path("/content/goes16_2da")

downloaded_path = kagglehub.dataset_download(
    handle=(
        "colvertgomez/"
        "goes16-wildfires-smoke-plumes-dataset"
    ),
    path="GOES16-wildfires-smoke-plumes-dataset-2da",
    output_dir=str(DOWNLOAD_DIR),
)

print("Downloaded folder:", downloaded_path)

KaggleApiHTTPError: 429 Client Error: Too Many Requests for url: https://api.kaggle.com/v1/datasets.DatasetApiService/DownloadDataset

In [ ]:
DATASET_ROOT = Path(downloaded_path)

print("Dataset root:", DATASET_ROOT)

for item in sorted(DATASET_ROOT.iterdir()):
    print(item.name)

In [ ]:
for path in DOWNLOAD_DIR.iterdir():
    print(
        path.name,
        f"{path.stat().st_size / 1024**3:.2f} GB"
    )

In [ ]:
DATASET_ROOT = (
    EXTRACT_DIR
    / "GOES16-wildfires-smoke-plumes-dataset-2da"
)

TRAIN_IMAGES = DATASET_ROOT / "train"
TRAIN_MASKS = DATASET_ROOT / "train_masks"

VALID_IMAGES = DATASET_ROOT / "valid"
VALID_MASKS = DATASET_ROOT / "valid_masks"

TEST_IMAGES = DATASET_ROOT / "test"
TEST_MASKS = DATASET_ROOT / "test_masks"

required_paths = [
    TRAIN_IMAGES,
    TRAIN_MASKS,
    VALID_IMAGES,
    VALID_MASKS,
    TEST_IMAGES,
    TEST_MASKS,
]

for path in required_paths:
    print(path, "->", path.exists())

assert all(path.exists() for path in required_paths), (
    "The expected dataset structure was not found."
)

In [ ]:
if not DATASET_ROOT.exists():
    print("Dataset missing: run download and extraction cells.")
else:
    print("Dataset already available. Skipping download.")

In [ ]:
import glob
print("Current directory files:")
display(os.listdir('.'))
print("\n--- Detailed Archive Directory Tree ---")
for root, dirs, files in os.walk('.'):
    level = root.replace('.', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:5]: # display up to first 5 files in each directory
        print(f"{subindent}{f}")

In [ ]:
import os

# Define paths based on the extracted contents
train_dir = 'train'
train_masks_dir = 'train_masks'
valid_dir = 'valid'
valid_masks_dir = 'valid_masks'
test_dir = 'test'
test_masks_dir = 'test_masks'

# Load files if the directories exist, otherwise default to empty lists
train_images = os.listdir(train_dir) if os.path.exists(train_dir) else []
train_masks = os.listdir(train_masks_dir) if os.path.exists(train_masks_dir) else []

valid_images = os.listdir(valid_dir) if os.path.exists(valid_dir) else []
valid_masks = os.listdir(valid_masks_dir) if os.path.exists(valid_masks_dir) else []

test_images = os.listdir(test_dir) if os.path.exists(test_dir) else []
test_masks = os.listdir(test_masks_dir) if os.path.exists(test_masks_dir) else []

# Print the counts
print(f"Train images: {len(train_images)}, Train masks: {len(train_masks)}")
print(f"Validation images: {len(valid_images)}, Validation masks: {len(valid_masks)}")
print(f"Test images: {len(test_images)}, Test masks: {len(test_masks)}")

In [ ]:
# Install unrar package to extract .rar format, then extract the downloaded dataset
!sudo apt-get install unrar -y
!unrar x dataset.rar

In [ ]:
pip install segmentation-models-pytorch albumentations opencv-python tqdm scikit-learn

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A

from tqdm import tqdm

import segmentation_models_pytorch as smp
from PIL import Image

In [ ]:
DATASET_PATH = "."

In [ ]:
TRAIN_IMAGES = os.path.join(DATASET_PATH, "train")
TRAIN_MASKS = os.path.join(DATASET_PATH, "train_masks")

VALID_IMAGES = os.path.join(DATASET_PATH, "valid")
VALID_MASKS = os.path.join(DATASET_PATH, "valid_masks")

TEST_IMAGES = os.path.join(DATASET_PATH, "test")
TEST_MASKS = os.path.join(DATASET_PATH, "test_masks")

In [ ]:
import numpy as np

images = sorted(os.listdir(TRAIN_IMAGES))
masks = sorted(os.listdir(TRAIN_MASKS))

mask = np.array(
    Image.open(
        os.path.join(TRAIN_MASKS, masks[0])
    )
)

print(mask.shape)
print(np.unique(mask))

In [ ]:
image = cv2.imread(
    os.path.join(TRAIN_IMAGES, images[0])
)

print(image.shape)

In [ ]:
from PIL import Image
import numpy as np

mask = np.array(
    Image.open(
        os.path.join(
            TRAIN_MASKS,
            sorted(os.listdir(TRAIN_MASKS))[0]
        )
    )
)

print(mask.dtype)
print(mask.min())
print(mask.max())
print(mask.mean())

In [ ]:
from PIL import Image

class SmokeDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))

        self.transform = A.Compose([
            A.Resize(128,128)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = cv2.imread(
            os.path.join(
                self.image_dir,
                self.images[idx]
            )
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = np.array(
            Image.open(
                os.path.join(
                    self.mask_dir,
                    self.masks[idx]
                )
            )
        )

        transformed = self.transform(
            image=image,
            mask=mask
        )

        image = transformed["image"]
        mask = transformed["mask"]

        image = image.astype(np.float32) / 255.0

        image = np.transpose(
            image,
            (2,0,1)
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        )

        mask = mask.unsqueeze(0)

        return image, mask

In [ ]:
dataset = SmokeDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS
)

image, mask = dataset[0]

print(image.shape)
print(mask.shape)

In [ ]:
train_dataset = SmokeDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS
)

valid_dataset = SmokeDataset(
    VALID_IMAGES,
    VALID_MASKS
)

test_dataset = SmokeDataset(
    TEST_IMAGES,
    TEST_MASKS
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="mit_b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print(device)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

outputs = model(images)

print("Images :", images.shape)
print("Masks  :", masks.shape)
print("Output :", outputs.shape)

In [ ]:
dice_loss = smp.losses.DiceLoss(mode="binary")
focal_loss = smp.losses.FocalLoss(mode="binary")

def loss_fn(preds, targets):
    return (
        0.7 * dice_loss(preds, targets)
        + 0.3 * focal_loss(preds, targets)
    )

In [ ]:
loss = loss_fn(outputs, masks)

print(loss.item())

In [ ]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fn(outputs, masks)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate(model, loader):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = loss_fn(outputs, masks)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
best_val = 999

for epoch in range(20):

    train_loss = train_epoch(
        model,
        train_loader
    )

    val_loss = validate(
        model,
        valid_loader
    )

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            "best_unet.pth"
        )

        print("Best model saved")

In [ ]:
val_loss = validate(
    model,
    valid_loader
)

print("Validation Loss:", val_loss)

In [ ]:
def iou_score(preds, masks):

    preds = (preds > 0.5).float()

    intersection = (preds * masks).sum()

    union = (preds + masks).sum() - intersection

    return ((intersection + 1e-7) /
            (union + 1e-7))

In [ ]:
model.eval()

ious = []

with torch.no_grad():

    for images, masks in valid_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        outputs = torch.sigmoid(outputs)

        score = iou_score(
            outputs,
            masks
        )

        ious.append(
            score.cpu().numpy()
        )

print("Mean IoU:", np.mean(ious))

In [ ]:
def f1_score(preds, masks):

    preds = (preds > 0.5).float()

    tp = (preds * masks).sum()

    precision = tp / (preds.sum() + 1e-7)

    recall = tp / (masks.sum() + 1e-7)

    f1 = (2 * precision * recall) / \
         (precision + recall + 1e-7)

    return f1

In [ ]:
model.eval()

f1s = []

with torch.no_grad():

    for images, masks in valid_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        outputs = torch.sigmoid(outputs)

        score = f1_score(
            outputs,
            masks
        )

        f1s.append(
            score.cpu().numpy()
        )

print("Mean F1:", np.mean(f1s))

In [ ]:
model.eval()

images, masks = next(iter(valid_loader))

image = images[0]
mask = masks[0]

with torch.no_grad():

    pred = model(
        image.unsqueeze(0).to(device)
    )

    pred = torch.sigmoid(pred)

pred = pred.squeeze().cpu().numpy()

image = image.permute(1,2,0).numpy()

mask = mask.squeeze().numpy()

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(image)
plt.title("Image")

plt.subplot(1,3,2)
plt.imshow(mask,cmap="gray")
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred,cmap="gray")
plt.title("Prediction")

plt.show()

In [ ]:
print(smp.__version__)